# 🌐 Notebook 05: Servidor Remoto vLLM + Túnel Cloudflare para a Interface TUI

Este notebook carrega o seu modelo **CyberLLM 32B real** salvo no Google Drive (`qwen32b_cyber_merged_bf16` ou `qwen32b_cyber_lora_final`), sobe o motor **vLLM OpenAI API** na GPU do Google Colab e cria um túnel HTTPS público gratuito via **Cloudflare Tunnel (`trycloudflare.com`)**.

Ao final da **Célula 3**, copie o comando `./run_tui.sh --api-base https://...trycloudflare.com/v1 --save` exibido na tela e cole no terminal do seu **Kali Linux**!

In [ ]:
# 1. Montar Google Drive e localizar seu modelo CyberLLM 32B treinado
from google.colab import drive
import os, torch

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/cyber_llm_project'
MERGED_PATH = os.path.join(DRIVE_BASE, 'exported_models/qwen32b_cyber_merged_bf16')
LORA_PATH = os.path.join(DRIVE_BASE, 'model_checkpoints/qwen32b_cyber_lora_final')

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
print(f'[+] GPU detectada: {torch.cuda.get_device_name(0) if vram_gb else "Nenhuma"} ({vram_gb:.1f} GB VRAM)')

if os.path.exists(MERGED_PATH):
    MODEL_ARGS = f'--model "{MERGED_PATH}" --served-model-name qwen32b-cyber'
    print(f'[✓] Modelo 32B BF16 consolidado encontrado: {MERGED_PATH}')
elif os.path.exists(LORA_PATH):
    MODEL_ARGS = (
        f'--model Qwen/Qwen2.5-Coder-32B-Instruct '
        f'--enable-lora --max-lora-rank 64 '
        f'--lora-modules qwen32b-cyber="{LORA_PATH}"'
    )
    print(f'[✓] Adaptadores LoRA encontrados: {LORA_PATH}')
else:
    MODEL_ARGS = '--model Qwen/Qwen2.5-Coder-32B-Instruct --served-model-name qwen32b-cyber'
    print('[!] Pesos no Drive não encontrados neste caminho; usando Qwen2.5-Coder-32B-Instruct base.')

# Ajusta precisão conforme a VRAM da sessão atual do Colab
if vram_gb >= 70:
    QUANT_ARGS = '--dtype bfloat16 --max-model-len 16384 --gpu-memory-utilization 0.92'
else:
    QUANT_ARGS = '--quantization bitsandbytes --load-format bitsandbytes --max-model-len 8192 --gpu-memory-utilization 0.92'


In [ ]:
# 2. Instalar vLLM + binário cloudflared (túnel gratuito sem cadastro)
!pip install -q vllm bitsandbytes
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('[✓] vLLM e Cloudflared prontos!')

In [ ]:
# 3. Iniciar vLLM na porta 8000 + Túnel HTTPS Público para sua TUI no Kali Linux
import subprocess, time, re, urllib.request

vllm_cmd = f'python3 -m vllm.entrypoints.openai.api_server {MODEL_ARGS} {QUANT_ARGS} --host 0.0.0.0 --port 8000'
print('[*] Iniciando motor vLLM (carregando pesos na VRAM, aguarde ~1-3 min)...')
vllm_log = open('/content/vllm_server.log', 'w')
vllm_proc = subprocess.Popen(vllm_cmd, shell=True, stdout=vllm_log, stderr=subprocess.STDOUT)

# Abre o túnel Cloudflare
cf_log = open('/content/cloudflared.log', 'w')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=cf_log, stderr=subprocess.STDOUT
)

public_url = None
for _ in range(30):
    time.sleep(1)
    txt = open('/content/cloudflared.log').read()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
    if m:
        public_url = m.group(0)
        break

# Aguarda o vLLM responder em /v1/models
ready = False
while vllm_proc.poll() is None:
    try:
        with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(4)

if ready and public_url:
    print('\n' + '='*78)
    print('🚀 CYBERLLM 32B REAL ONLINE NA GPU DO COLAB!')
    print(f'🔗 Endpoint Público OpenAI/vLLM: {public_url}/v1')
    print('\n👉 Rode isto no terminal do seu Kali Linux para conectar a TUI na IA REAL:')
    print(f'./run_tui.sh --api-base {public_url}/v1 --model qwen32b-cyber --save')
    print('='*78)
    # Mantém a célula viva enquanto você usa a TUI
    try:
        vllm_proc.wait()
    except KeyboardInterrupt:
        vllm_proc.terminate()
        cf_proc.terminate()
else:
    print('[!] Falha ao subir o vLLM. Últimas linhas do log:')
    print(open('/content/vllm_server.log').read()[-2000:])